# 🧠 SkinTwin AtomSpace Integration

## Overview

This lab demonstrates the integration of OpenCog AtomSpace with Azure API Management for dermatology knowledge representation. It shows how domain-specific ontologies for skin conditions, treatments, and anatomical structures are managed through the RegimAI Gateway.

![SkinTwin AtomSpace Flow](../../images/skintwin-atomspace.gif)

The lab showcases:
- **Knowledge Representation**: Dermatology-specific concepts in AtomSpace format
- **Truth Value Management**: Uncertainty handling for medical knowledge
- **Cognitive Routing**: Intelligent request routing based on knowledge complexity
- **Clinical Integration**: Evidence-based medical reasoning

## Prerequisites

- [Python 3.12 or later](https://www.python.org/) installed
- [Azure subscription](https://azure.microsoft.com/free/) with appropriate permissions
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and configured
- **Medical Domain Knowledge** recommended for interpreting results
- **OpenCog Understanding** helpful but not required

## Architecture

```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   Client App    │───▶│  RegimAI Gateway │───▶│  Azure OpenAI   │
└─────────────────┘    └──────────────────┘    └─────────────────┘
                                │
                                ▼
                       ┌──────────────────┐
                       │ SkinTwin AtomSpace│
                       │ ┌──────────────┐ │
                       │ │ Dermatology  │ │
                       │ │ Ontology     │ │
                       │ └──────────────┘ │
                       │ ┌──────────────┐ │
                       │ │ Treatment    │ │
                       │ │ Knowledge    │ │
                       │ └──────────────┘ │
                       │ ┌──────────────┐ │
                       │ │ Evidence     │ │
                       │ │ Base         │ │
                       │ └──────────────┘ │
                       └──────────────────┘
```

## Lab Configuration

The following configuration sets up the SkinTwin AtomSpace integration:

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "uksouth"

# SkinTwin-specific configuration
cognitive_config = {
    "atomspace_enabled": True,
    "pln_reasoning": True,
    "medical_ontology": "dermatology_v2.0",
    "evidence_validation": True,
    "clinical_compliance": "HIPAA"
}

# Azure OpenAI models for dermatology
openai_config = [
    {
        "name": "dermatology-gpt4", 
        "location": "uksouth",
        "specialization": "medical_dermatology"
    }
]

# Model configuration for medical use
models_config = [
    {
        "name": "gpt-4-turbo-dermatology", 
        "publisher": "OpenAI", 
        "version": "2024-04-09", 
        "sku": "GlobalStandard", 
        "capacity": 10,
        "medical_certified": True
    }
]

apim_sku = 'StandardV2'  # Higher tier for medical compliance
apim_subscriptions_config = [
    {
        "name": "dermatology-subscription", 
        "displayName": "SkinTwin Dermatology Access",
        "compliance": "medical"
    }
]

cognitive_api_path = "cognitive/atomspace"

## Deploying the Lab

In [ ]:
# Deploy the infrastructure
deployment_result = utils.deploy_lab(
    deployment_name=deployment_name,
    resource_group_name=resource_group_name,
    resource_group_location=resource_group_location,
    openai_config=openai_config,
    models_config=models_config,
    apim_sku=apim_sku,
    apim_subscriptions_config=apim_subscriptions_config,
    cognitive_config=cognitive_config
)

print("✅ SkinTwin AtomSpace lab deployed successfully!")
print(f"🌐 Gateway URL: {deployment_result['gateway_url']}")
print(f"🧠 AtomSpace Endpoint: {deployment_result['gateway_url']}/{cognitive_api_path}")

## SkinTwin Knowledge Ontology

The dermatology ontology includes structured medical knowledge:

In [ ]:
# Dermatology knowledge structure for AtomSpace
dermatology_ontology = {
    "concepts": {
        # Anatomical structures
        "skin_anatomy": ["epidermis", "dermis", "hypodermis", "hair_follicle", "sebaceous_gland"],
        
        # Skin conditions
        "conditions": ["acne_vulgaris", "psoriasis", "atopic_dermatitis", "rosacea", "melanoma"],
        
        # Treatments
        "treatments": ["topical_retinoids", "antibiotics", "phototherapy", "immunosuppressants"],
        
        # Active ingredients
        "ingredients": ["tretinoin", "benzoyl_peroxide", "salicylic_acid", "hydroquinone", "vitamin_c"]
    },
    
    "relationships": [
        {
            "type": "IsA",
            "source": "acne_vulgaris", 
            "target": "inflammatory_skin_condition",
            "truth_value": {"strength": 0.95, "confidence": 0.9}
        },
        {
            "type": "Treats", 
            "source": "tretinoin", 
            "target": "acne_vulgaris",
            "truth_value": {"strength": 0.85, "confidence": 0.92}
        },
        {
            "type": "LocatedIn",
            "source": "sebaceous_gland",
            "target": "dermis", 
            "truth_value": {"strength": 0.98, "confidence": 0.95}
        }
    ],
    
    "evidence": {
        "tretinoin_acne_efficacy": {
            "studies": ["randomized_controlled_trial_2023", "meta_analysis_2022"],
            "strength": 0.85,
            "confidence": 0.92,
            "regulatory_approval": "FDA_approved"
        }
    }
}

print("📚 Dermatology ontology structure:")
print(json.dumps(dermatology_ontology, indent=2))

## Testing AtomSpace Integration

In [ ]:
import requests

# Gateway endpoints
gateway_url = deployment_result['gateway_url']
atomspace_endpoint = f"{gateway_url}/{cognitive_api_path}"

# Test AtomSpace query
def test_atomspace_query(concept, relationship_type=None):
    headers = {
        'X-API-Key': deployment_result['api_key'],
        'Content-Type': 'application/json'
    }
    
    query_data = {
        "concept": concept,
        "relationship_type": relationship_type,
        "include_evidence": True,
        "medical_context": True
    }
    
    response = requests.post(f"{atomspace_endpoint}/query", 
                           headers=headers, 
                           json=query_data)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"❌ Query failed: {response.status_code}")
        return None

# Test queries
print("🔍 Testing AtomSpace queries...")

# Query for acne treatment relationships
acne_treatments = test_atomspace_query("acne_vulgaris", "Treats")
print("Acne treatments from AtomSpace:")
print(json.dumps(acne_treatments, indent=2))

# Query for ingredient mechanisms
ingredient_mechanisms = test_atomspace_query("tretinoin", "MechanismOfAction")
print("\\nTretinoin mechanisms:")
print(json.dumps(ingredient_mechanisms, indent=2))

## Cognitive Reasoning with PLN

In [ ]:
# Test probabilistic logic reasoning
def test_pln_reasoning(premise, conclusion):
    headers = {
        'X-API-Key': deployment_result['api_key'],
        'Content-Type': 'application/json'
    }
    
    reasoning_data = {
        "premise": premise,
        "conclusion": conclusion,
        "reasoning_type": "clinical_inference",
        "evidence_required": True,
        "uncertainty_handling": True
    }
    
    response = requests.post(f"{gateway_url}/cognitive/reasoning", 
                           headers=headers, 
                           json=reasoning_data)
    
    return response.json() if response.status_code == 200 else None

# Test clinical reasoning
print("🔬 Testing PLN clinical reasoning...")

# Reason about treatment efficacy
reasoning_result = test_pln_reasoning(
    premise="Patient has moderate acne vulgaris",
    conclusion="Tretinoin 0.1% cream will be effective"
)

print("Clinical reasoning result:")
print(json.dumps(reasoning_result, indent=2))

## Cognitive-Aware Routing

In [ ]:
# Test cognitive complexity assessment for routing
def test_cognitive_routing(medical_query):
    headers = {
        'X-API-Key': deployment_result['api_key'],
        'Content-Type': 'application/json'
    }
    
    routing_data = {
        "query": medical_query,
        "assess_complexity": True,
        "medical_context": True,
        "routing_recommendation": True
    }
    
    response = requests.post(f"{gateway_url}/cognitive/analyze", 
                           headers=headers, 
                           json=routing_data)
    
    return response.json() if response.status_code == 200 else None

# Test different complexity levels
test_queries = [
    "What is the best moisturizer for dry skin?",  # Low complexity
    "Can you explain the pathophysiology of acne and recommend a treatment protocol?",  # High complexity
    "I have red bumps on my face, what should I do?"  # Medium complexity
]

print("🎯 Testing cognitive-aware routing...")

for query in test_queries:
    routing_result = test_cognitive_routing(query)
    print(f"\\nQuery: {query}")
    print(f"Complexity: {routing_result['complexity']['level']}")
    print(f"Recommended endpoint: {routing_result['routing']['recommended_endpoint']}")
    print(f"Processing priority: {routing_result['routing']['processing_priority']}")

## Medical Compliance Validation

In [ ]:
# Test medical compliance features
def test_medical_compliance(clinical_advice):
    headers = {
        'X-API-Key': deployment_result['api_key'],
        'Content-Type': 'application/json'
    }
    
    compliance_data = {
        "content": clinical_advice,
        "compliance_checks": [
            "medical_accuracy",
            "evidence_based",
            "appropriate_disclaimers",
            "no_diagnosis_claims"
        ],
        "regulatory_framework": "FDA_guidance"
    }
    
    response = requests.post(f"{gateway_url}/compliance/validate", 
                           headers=headers, 
                           json=compliance_data)
    
    return response.json() if response.status_code == 200 else None

# Test compliance validation
test_advice = "Based on your symptoms, you likely have acne. I recommend using tretinoin cream."

print("🛡️ Testing medical compliance validation...")
compliance_result = test_medical_compliance(test_advice)
print("Compliance validation result:")
print(json.dumps(compliance_result, indent=2))

## Evidence-Based Recommendations

In [ ]:
# Generate evidence-based skincare recommendations
def get_evidence_based_recommendation(skin_concern, skin_type):
    headers = {
        'X-API-Key': deployment_result['api_key'],
        'Content-Type': 'application/json'
    }
    
    request_data = {
        "skin_concern": skin_concern,
        "skin_type": skin_type,
        "evidence_level": "clinical_studies",
        "include_mechanism": True,
        "safety_considerations": True
    }
    
    response = requests.post(f"{gateway_url}/agents/evidence-based-consultant", 
                           headers=headers, 
                           json=request_data)
    
    return response.json() if response.status_code == 200 else None

# Test evidence-based recommendations
print("📊 Testing evidence-based recommendations...")

recommendation = get_evidence_based_recommendation(
    skin_concern="hyperpigmentation",
    skin_type="combination"
)

print("Evidence-based recommendation:")
print(json.dumps(recommendation, indent=2))

## AtomSpace Knowledge Exploration

In [ ]:
# Explore the knowledge graph structure
def explore_knowledge_graph():
    headers = {
        'X-API-Key': deployment_result['api_key']
    }
    
    response = requests.get(f"{atomspace_endpoint}/stats", headers=headers)
    stats = response.json() if response.status_code == 200 else {}
    
    print("🧠 AtomSpace Knowledge Statistics:")
    print(f"   Total atoms: {stats.get('total_atoms', 'N/A')}")
    print(f"   Concept nodes: {stats.get('concept_nodes', 'N/A')}")
    print(f"   Relationship links: {stats.get('relationship_links', 'N/A')}")
    print(f"   Evidence nodes: {stats.get('evidence_nodes', 'N/A')}")
    print(f"   Medical concepts: {stats.get('medical_concepts', 'N/A')}")
    
    # Get top concepts by importance
    top_concepts = requests.get(f"{atomspace_endpoint}/top-concepts", headers=headers)
    if top_concepts.status_code == 200:
        print("\\n📈 Most Important Concepts:")
        for concept in top_concepts.json()['concepts'][:10]:
            print(f"   {concept['name']}: {concept['importance']:.3f}")

explore_knowledge_graph()

## Cleanup

In [ ]:
# Clean up resources
print("🧹 Cleaning up lab resources...")

cleanup_result = utils.cleanup_lab(
    resource_group_name=resource_group_name,
    deployment_name=deployment_name
)

if cleanup_result:
    print("✅ Lab resources cleaned up successfully!")
else:
    print("⚠️ Some resources may need manual cleanup")

## Key Takeaways

This lab demonstrates:

1. **Knowledge Representation**: How dermatological knowledge is structured in AtomSpace format with truth values and evidence
2. **Cognitive Routing**: Intelligent request routing based on medical complexity analysis  
3. **Clinical Reasoning**: PLN-based probabilistic reasoning for medical decision support
4. **Evidence Integration**: Linking clinical evidence to knowledge representations
5. **Medical Compliance**: Ensuring AI outputs meet healthcare regulatory requirements

## Next Steps

- Explore [PLN Clinical Reasoning](../pln-clinical-reasoning/pln-clinical-reasoning.ipynb) for advanced medical inference
- Try [Medical Content Safety](../medical-content-safety/medical-content-safety.ipynb) for healthcare-specific content filtering
- Experiment with [HIPAA Compliance](../hipaa-compliance/hipaa-compliance.ipynb) for health data protection

## Medical Disclaimer

This lab is for educational and development purposes only. All medical knowledge representations and AI-generated recommendations must be reviewed and validated by qualified healthcare professionals before any clinical application. This system does not replace professional medical diagnosis or treatment.